# 🖼️ 생성형 AI를 활용한 인덱스 데이터 증강

이 실습에서는 **GPT**의 Vision 기능을 사용하여 제품 이미지를 분석하고, 분석 결과를 기존 인덱스 데이터에 추가하여 **검색 품질을 향상**시킵니다.

## 📋 학습 목표

1. **이미지 분석**: GPT Vision으로 제품 이미지 특징 추출
2. **콘텐츠 증강**: 기존 텍스트에 이미지 분석 결과 추가
3. **벡터 업데이트**: 증강된 콘텐츠로 content_vector 재생성
4. **인덱스 업데이트**: Azure AI Search 인덱스에 새 벡터 업로드

## 🎯 비즈니스 목표

기존 검색은 **텍스트 정보(제품명, 브랜드, 카테고리)**만 활용했습니다.  
이미지 분석을 통해 다음과 같은 **추가 정보**를 확보할 수 있습니다:

- 제품 색상, 디자인 특징
- 소재/재질 정보
- 사용 시나리오/용도
- 마케팅 어필 포인트

---

## 🔄 데이터 증강 프로세스

```
기존 content_text (벡터화 대상):
┌────────────────────────────────┐
│ **제품명**: [title]            │
│ **브랜드**: [brand]            │
│ **카테고리**: [category]       │
└────────────────────────────────┘
                ↓
        🖼️ 이미지 분석 (GPT Vision)
                ↓
증강된 content_text:
┌────────────────────────────────┐
│ **제품명**: [title]            │
│ **브랜드**: [brand]            │
│ **카테고리**: [category]       │
│ **제품특징**: [AI 분석 결과]  │  ← 추가!
└────────────────────────────────┘
                ↓
        📊 임베딩 재생성
                ↓
        ✅ content_vector 업데이트
```

---
## 1️⃣ 환경 설정

In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from openai import AzureOpenAI

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")

# Azure OpenAI 설정
OPENAI_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
OPENAI_KEY = os.getenv("FOUNDRY_PROJECT_KEY")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")  # GPT
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# 클라이언트 초기화
credential = AzureKeyCredential(SEARCH_ADMIN_KEY)
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=credential
)

openai_client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_KEY,
    api_version=API_VERSION
)

print("✅ 환경 설정 완료")
print(f"   Search Endpoint: {SEARCH_ENDPOINT}")
print(f"   Index: {INDEX_NAME}")
print(f"   Embedding Model: {EMBEDDING_DEPLOYMENT}")
print(f"   Vision Model: {CHAT_DEPLOYMENT}")

✅ 환경 설정 완료
   Search Endpoint: https://foundryiq-aisearch-260114-changjuahn.search.windows.net
   Index: products-index
   Embedding Model: text-embedding-3-small
   Vision Model: gpt-4.1-mini


---
## 2️⃣ 데이터 로드

CSV 파일에서 제품 데이터와 이미지 URL을 로드합니다.

In [2]:
# CSV 파일 경로
csv_file = "../00-data/sample_data.csv"

# CSV 읽기
df = pd.read_csv(csv_file)

print(f"✅ CSV 파일 로드 완료")
print(f"   총 데이터 수: {len(df)}개")
print(f"   컬럼: {list(df.columns)}")

# 이미지 URL 샘플 확인
print(f"\n🖼️ 이미지 URL 샘플:")
for i in range(3):
    print(f"   {i+1}. {df.iloc[i]['title'][:30]}...")
    print(f"      URL: {df.iloc[i]['image_link']}")

✅ CSV 파일 로드 완료
   총 데이터 수: 247개
   컬럼: ['id', 'title', 'brand', 'category', 'normal_price', 'image_link', 'review']

🖼️ 이미지 URL 샘플:
   1. 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아...
      URL: https://foundryiq-image-gallery.azurewebsites.net/images/2119554205_0_600.jpg
   2. [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA...
      URL: https://foundryiq-image-gallery.azurewebsites.net/images/2130369107_0_600.jpg
   3. 블루독베이비[hu] WH)블루독남아손수건SET 4117...
      URL: https://foundryiq-image-gallery.azurewebsites.net/images/2138687276_0_600.jpg


---
## 3️⃣ 이미지 분석 함수 정의

GPT의 Vision 기능을 사용하여 제품 이미지를 분석하고, **판매자/마케팅 전문가 관점**에서 특징을 추출합니다.

In [ ]:
def analyze_product_image(image_url, title, category, max_retries=3):
    """
    GPT Vision으로 제품 이미지 분석
    
    Args:
        image_url: 제품 이미지 URL
        title: 제품명 (컨텍스트 제공)
        category: 카테고리 (컨텍스트 제공)
        max_retries: 최대 재시도 횟수
        
    Returns:
        이미지 분석 결과 텍스트
    """
    
    system_prompt = """당신은 전문 상품 마케터이자 쇼핑몰 판매 전문가입니다.
제품 이미지를 보고 고객에게 어필할 수 있는 핵심 특징을 분석해주세요.

다음 관점에서 분석하세요:
1. 색상 및 디자인 특징
2. 소재/재질 (이미지에서 유추 가능한 경우)
3. 사용 시나리오/용도
4. 타겟 고객층
5. 마케팅 어필 포인트

응답 형식:
- 간결하고 명확하게 3-5문장으로 작성
- 고객이 검색할 만한 키워드를 자연스럽게 포함
- 마크다운 없이 일반 텍스트로 작성"""
    
    user_prompt = f"""다음 제품 이미지를 분석해주세요.

제품명: {title}
카테고리: {category}

이 제품의 시각적 특징과 마케팅 포인트를 분석해주세요."""
    
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.create(
                model=CHAT_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": user_prompt},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": image_url,
                                    "detail": "low"  # 비용 절감을 위해 low detail 사용
                                }
                            }
                        ]
                    }
                ],
                max_tokens=300,
                temperature=0.3  # 일관된 결과를 위해 낮은 temperature
            )
            
            return response.choices[0].message.content.strip()
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"⚠️  이미지 분석 에러, {wait_time}초 후 재시도... ({attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"❌ 이미지 분석 실패: {str(e)[:150]}...")
                return None


print("✅ 이미지 분석 함수 정의 완료")

✅ 이미지 분석 함수 정의 완료


### 이미지 분석 테스트

몇 개의 샘플 제품으로 이미지 분석 기능을 테스트합니다.

In [4]:
# 테스트: 첫 번째 제품 이미지 분석
test_row = df.iloc[0]

print(f"🧪 이미지 분석 테스트")
print(f"   제품명: {test_row['title'][:50]}...")
print(f"   카테고리: {test_row['category']}")
print(f"   이미지 URL: {test_row['image_link']}")
print("\n분석 중...\n")

analysis_result = analyze_product_image(
    image_url=test_row['image_link'],
    title=test_row['title'],
    category=test_row['category']
)

if analysis_result:
    print("✅ 이미지 분석 결과:")
    print("-" * 60)
    print(analysis_result)
    print("-" * 60)
else:
    print("❌ 이미지 분석 실패")

🧪 이미지 분석 테스트
   제품명: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   카테고리: 유아동
   이미지 URL: https://foundryiq-image-gallery.azurewebsites.net/images/2119554205_0_600.jpg

분석 중...

✅ 이미지 분석 결과:
------------------------------------------------------------
이 제품은 부드럽고 파스텔톤의 오렌지, 연두, 하늘색 등 아기에게 편안한 색상으로 디자인되어 신생아 시각 자극에 적합합니다. 소재는 아기 손에 안전한 부드러운 플라스틱과 천 재질이 혼합되어 있어 촉감 놀이와 안전한 사용이 가능합니다. 신생아 백일 선물용으로 적합하며, 다양한 형태의 딸랑이와 촉감 완구가 포함되어 아기의 감각 발달과 흥미 유발에 효과적입니다. 타겟은 신생아 부모 및 가족으로, 안전하고 감각 발달에 좋은 유아용 딸랑이 세트를 찾는 고객에게 어필할 수 있습니다. ‘신생아 딸랑이 세트’, ‘백일 선물 추천’, ‘감각 발달 완구’ 키워드로 마케팅하기 좋습니다.
------------------------------------------------------------


In [5]:
# 다른 카테고리 제품도 테스트
# 스포츠/레져 카테고리 제품 찾기
sports_row = df[df['category'] == '스포츠/레져'].iloc[0]

print(f"🧪 이미지 분석 테스트 (스포츠/레져)")
print(f"   제품명: {sports_row['title'][:50]}...")
print(f"   이미지 URL: {sports_row['image_link']}")
print("\n분석 중...\n")

analysis_result_2 = analyze_product_image(
    image_url=sports_row['image_link'],
    title=sports_row['title'],
    category=sports_row['category']
)

if analysis_result_2:
    print("✅ 이미지 분석 결과:")
    print("-" * 60)
    print(analysis_result_2)
    print("-" * 60)

🧪 이미지 분석 테스트 (스포츠/레져)
   제품명: [젝시믹스] V업 3D 플러스 레깅스 1+1 (XP9156T)...
   이미지 URL: https://foundryiq-image-gallery.azurewebsites.net/images/2210550987_0_600.jpg

분석 중...

✅ 이미지 분석 결과:
------------------------------------------------------------
이 제품은 심플하면서도 세련된 다크 그레이와 블랙 컬러의 3D 입체 패턴 레깅스로, 몸매 라인을 자연스럽게 살려주는 디자인이 특징입니다. 신축성 좋은 소재로 운동 시 편안한 착용감을 제공하며, 조깅이나 요가, 헬스 등 다양한 스포츠 활동에 적합합니다. 특히 ‘V업 3D’ 기능으로 복부와 힙 라인을 탄탄하게 잡아주어 슬림한 실루엣을 연출해 줍니다. 활동적인 20~30대 여성 운동복으로 추천하며, 1+1 구성으로 가성비를 강조한 마케팅 포인트가 돋보입니다. 키워드: 스포츠 레깅스, 3D 입체 레깅스, 운동복, 요가 팬츠, 슬림핏 레깅스.
------------------------------------------------------------


---
## 4️⃣ 증강 콘텐츠 생성 함수

기존 content_text에 이미지 분석 결과를 추가하여 **증강된 콘텐츠**를 생성합니다.

In [6]:
def create_enriched_content(row, image_analysis):
    """
    기존 content_text에 이미지 분석 결과를 추가하여 증강된 콘텐츠 생성
    
    Args:
        row: DataFrame row (title, brand, category)
        image_analysis: GPT Vision 분석 결과
        
    Returns:
        증강된 마크다운 형식의 콘텐츠
    """
    content_parts = []
    
    # 기존 정보 (제품명, 브랜드, 카테고리)
    if pd.notna(row['title']) and str(row['title']).strip():
        content_parts.append(f"**제품명**: {row['title']}")
    
    if pd.notna(row['brand']) and str(row['brand']).strip():
        content_parts.append(f"**브랜드**: {row['brand']}")
    
    if pd.notna(row['category']) and str(row['category']).strip():
        content_parts.append(f"**카테고리**: {row['category']}")
    
    # 이미지 분석 결과 추가 (증강!)
    if image_analysis and isinstance(image_analysis, str) and image_analysis.strip():
        content_parts.append(f"**제품특징**: {image_analysis}")
    
    return "\n".join(content_parts) if content_parts else "정보 없음"


# 테스트
print("🧪 증강 콘텐츠 테스트")
print("\n" + "="*60)
print("[기존 content_text]")
print("-"*60)
original_content = f"""**제품명**: {test_row['title']}
**브랜드**: {test_row['brand']}
**카테고리**: {test_row['category']}"""
print(original_content)

print("\n" + "="*60)
print("[증강된 content_text]")
print("-"*60)
enriched_content = create_enriched_content(test_row, analysis_result)
print(enriched_content)
print("="*60)

🧪 증강 콘텐츠 테스트

[기존 content_text]
------------------------------------------------------------
**제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동

[증강된 content_text]
------------------------------------------------------------
**제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동
**제품특징**: 이 제품은 부드럽고 파스텔톤의 오렌지, 연두, 하늘색 등 아기에게 편안한 색상으로 디자인되어 신생아 시각 자극에 적합합니다. 소재는 아기 손에 안전한 부드러운 플라스틱과 천 재질이 혼합되어 있어 촉감 놀이와 안전한 사용이 가능합니다. 신생아 백일 선물용으로 적합하며, 다양한 형태의 딸랑이와 촉감 완구가 포함되어 아기의 감각 발달과 흥미 유발에 효과적입니다. 타겟은 신생아 부모 및 가족으로, 안전하고 감각 발달에 좋은 유아용 딸랑이 세트를 찾는 고객에게 어필할 수 있습니다. ‘신생아 딸랑이 세트’, ‘백일 선물 추천’, ‘감각 발달 완구’ 키워드로 마케팅하기 좋습니다.


---
## 5️⃣ 전체 데이터 이미지 분석 및 콘텐츠 증강

⚠️ **주의**: 전체 247개 제품을 분석하면 시간과 비용이 소요됩니다.  
테스트 목적으로 일부 데이터만 처리하거나, 전체 처리를 선택할 수 있습니다.

In [7]:
# 처리할 데이터 수 선택
# 기본: 앞 10개는 임베딩/업데이트/비교, 뒤 10개는 홀드아웃(임베딩/업데이트 안 함)
HEAD_COUNT = 10
TAIL_COUNT = 10

# 필요 시 전체 처리하려면 아래를 사용하세요.
# df_sample = df.copy()

# 실험 대상(임베딩/업데이트/비교): 앞쪽 HEAD_COUNT
df_sample = df.head(HEAD_COUNT).copy()

# 홀드아웃: 뒤쪽 TAIL_COUNT (임베딩/업데이트 하지 않음, 필요 시 별도 활용)
df_holdout_tail = df.tail(TAIL_COUNT).copy()

print(f"🚀 이미지 분석 시작: {len(df_sample)}개 제품 (실험 대상)")
print(f"   포함: 앞 {HEAD_COUNT}개")
print(f"⏱️ 예상 소요 시간: 약 {len(df_sample) * 3}초 ~ {len(df_sample) * 5}초\n")

🚀 이미지 분석 시작: 10개 제품 (실험 대상)
   포함: 앞 10개
⏱️ 예상 소요 시간: 약 30초 ~ 50초



In [8]:
# 이미지 분석 실행
image_analyses = []
success_count = 0
fail_count = 0

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="이미지 분석"):
    analysis = analyze_product_image(
        image_url=row['image_link'],
        title=row['title'],
        category=row['category']
    )
    
    if analysis:
        success_count += 1
    else:
        fail_count += 1
        analysis = ""  # 실패 시 빈 문자열
    
    image_analyses.append(analysis)
    
    # Rate limit 방지
    time.sleep(1)

df_sample['image_analysis'] = image_analyses

print(f"\n✅ 이미지 분석 완료!")
print(f"   성공: {success_count}개")
print(f"   실패: {fail_count}개")

이미지 분석: 100%|██████████| 10/10 [00:48<00:00,  4.85s/it]


✅ 이미지 분석 완료!
   성공: 10개
   실패: 0개


In [9]:
# 증강된 콘텐츠 생성
print("📝 증강된 콘텐츠 생성 중...")

enriched_contents = []
for idx, row in df_sample.iterrows():
    enriched = create_enriched_content(row, row['image_analysis'])
    enriched_contents.append(enriched)

df_sample['enriched_content'] = enriched_contents

print(f"✅ 증강된 콘텐츠 생성 완료: {len(enriched_contents)}개\n")

# 샘플 확인
print("📄 증강된 콘텐츠 샘플:")
print("="*80)
print(df_sample.iloc[0]['enriched_content'])
print("="*80)

📝 증강된 콘텐츠 생성 중...
✅ 증강된 콘텐츠 생성 완료: 10개

📄 증강된 콘텐츠 샘플:
**제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동
**제품특징**: 이 제품은 부드럽고 파스텔톤의 연한 녹색, 파랑, 주황색 등 아기 눈에 편안한 색상 조합으로 디자인되어 신생아에게 적합합니다. 소재는 아기 손에 안전한 무독성 플라스틱과 부드러운 천 재질이 혼합되어 있어 촉감과 안전성을 모두 고려한 점이 돋보입니다. 신생아 백일 선물용으로 적합하며, 시각적 자극과 촉각 발달을 돕는 딸랑이와 치발기 세트로 활용도가 높습니다. 타겟 고객은 신생아 부모 및 친지로, 안전하고 감각 발달에 좋은 유아용 장난감을 찾는 이들에게 어필할 수 있습니다. ‘신생아 딸랑이’, ‘백일 선물’, ‘안전한 유아 장난감’ 키워드로 마케팅하면 효과적입니다.


---
## 6️⃣ 임베딩 생성 및 벡터 업데이트

증강된 콘텐츠를 임베딩하여 content_vector를 업데이트합니다.

In [10]:
def get_embeddings_batch(texts, model=EMBEDDING_DEPLOYMENT, max_retries=3):
    """
    텍스트 리스트를 벡터로 변환 (배치 처리)
    """
    for attempt in range(max_retries):
        try:
            # 전처리
            processed_texts = []
            for text in texts:
                if not text or not isinstance(text, str):
                    text = "정보 없음"
                text = ' '.join(text.strip().split())
                processed_texts.append(text)
            
            # 임베딩 생성
            response = openai_client.embeddings.create(
                input=processed_texts,
                model=model
            )
            
            return [item.embedding for item in response.data]
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"⚠️  에러 발생, {wait_time}초 후 재시도...")
                time.sleep(wait_time)
            else:
                print(f"❌ 임베딩 생성 실패: {str(e)}")
                return [None] * len(texts)


print("✅ 임베딩 함수 정의 완료")

✅ 임베딩 함수 정의 완료


In [11]:
# 증강된 콘텐츠 임베딩 생성
BATCH_SIZE = 16

print(f"🚀 증강된 콘텐츠 임베딩 생성 중...")
print(f"   배치 크기: {BATCH_SIZE}")

enriched_vectors = []
for i in tqdm(range(0, len(df_sample), BATCH_SIZE), desc="임베딩 생성"):
    batch = df_sample['enriched_content'].iloc[i:i+BATCH_SIZE].tolist()
    vectors = get_embeddings_batch(batch)
    enriched_vectors.extend(vectors)
    time.sleep(0.5)  # Rate limit 방지

df_sample['enriched_vector'] = enriched_vectors

# 결과 확인
success_count = sum(1 for v in enriched_vectors if v is not None)
print(f"\n✅ 임베딩 생성 완료!")
print(f"   성공: {success_count}/{len(df_sample)}개")
if success_count > 0:
    print(f"   벡터 차원: {len(enriched_vectors[0]) if enriched_vectors[0] else 0}")

🚀 증강된 콘텐츠 임베딩 생성 중...
   배치 크기: 16


임베딩 생성: 100%|██████████| 1/1 [00:01<00:00,  1.86s/it]


✅ 임베딩 생성 완료!
   성공: 10/10개
   벡터 차원: 1536


In [21]:

# 📌 중요: 비교 분석을 위해 기존 벡터를 먼저 저장 (실험군 + 대조군)
print("💾 기존 벡터 백업 (비교 분석용)...")
print(f"📌 실험군(df_sample) + 대조군(df_holdout_tail) 모두 백업")
print()

# 실험군 기존 벡터 백업
print("1️⃣ 실험군 기존 벡터 조회...")
existing_vectors = []
for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="실험군 벡터 조회"):
    try:
        doc = search_client.get_document(key=str(row['id']))
        if 'content_vector' in doc and doc['content_vector'] is not None:
            existing_vectors.append(doc['content_vector'])
        else:
            existing_vectors.append(None)
    except Exception as e:
        print(f"⚠️  문서 조회 실패 (ID={row['id']}): {str(e)[:40]}")
        existing_vectors.append(None)

df_sample['existing_vector'] = existing_vectors
sample_success = sum(1 for v in existing_vectors if v is not None)
print(f"✅ 실험군 백업 완료: {sample_success}/{len(df_sample)}개")

# 대조군 기존 벡터 백업
print("\n2️⃣ 대조군 기존 벡터 조회...")
holdout_existing_vectors = []
for idx, row in tqdm(df_holdout_tail.iterrows(), total=len(df_holdout_tail), desc="대조군 벡터 조회"):
    try:
        doc = search_client.get_document(key=str(row['id']))
        if 'content_vector' in doc and doc['content_vector'] is not None:
            holdout_existing_vectors.append(doc['content_vector'])
        else:
            holdout_existing_vectors.append(None)
    except Exception as e:
        print(f"⚠️  문서 조회 실패 (ID={row['id']}): {str(e)[:40]}")
        holdout_existing_vectors.append(None)

df_holdout_tail['existing_vector'] = holdout_existing_vectors
holdout_success = sum(1 for v in holdout_existing_vectors if v is not None)
print(f"✅ 대조군 백업 완료: {holdout_success}/{len(df_holdout_tail)}개")

# 전체 결과
print(f"\n{'='*60}")
print(f"✅ 전체 기존 벡터 백업 완료!")
print(f"   실험군: {sample_success}/{len(df_sample)}개")
print(f"   대조군: {holdout_success}/{len(df_holdout_tail)}개")
print(f"{'='*60}")


💾 기존 벡터 백업 (비교 분석용)...
📌 실험군(df_sample) + 대조군(df_holdout_tail) 모두 백업

1️⃣ 실험군 기존 벡터 조회...


실험군 벡터 조회: 100%|██████████| 10/10 [00:01<00:00,  8.21it/s]


✅ 실험군 백업 완료: 10/10개

2️⃣ 대조군 기존 벡터 조회...


대조군 벡터 조회: 100%|██████████| 10/10 [00:00<00:00, 20.33it/s]

✅ 대조군 백업 완료: 10/10개

✅ 전체 기존 벡터 백업 완료!
   실험군: 10/10개
   대조군: 10/10개


---
## 7️⃣ Azure AI Search 인덱스 업데이트

증강된 벡터를 인덱스의 `content_vector` 필드에 업데이트합니다.

In [13]:
# 업로드용 문서 준비
documents = []

for idx, row in df_sample.iterrows():
    if row['enriched_vector'] is not None:
        doc = {
            "id": str(row['id']),
            "content_vector": row['enriched_vector']  # 증강된 벡터로 업데이트
        }
        documents.append(doc)

print(f"📤 업로드 준비: {len(documents)}개 문서")
print(f"   업로드할 필드: id, content_vector (증강된 벡터)")

📤 업로드 준비: 10개 문서
   업로드할 필드: id, content_vector (증강된 벡터)


In [14]:
# 인덱스에 업로드
try:
    result = search_client.merge_or_upload_documents(documents=documents)
    
    # 결과 확인
    success = sum(1 for item in result if item.succeeded)
    failed = sum(1 for item in result if not item.succeeded)
    
    print(f"\n{'='*60}")
    print(f"✅ 인덱스 업데이트 완료!")
    print(f"{'='*60}")
    print(f"   성공: {success}개")
    print(f"   실패: {failed}개")
    
    if failed > 0:
        print("\n⚠️ 실패한 문서:")
        for item in result:
            if not item.succeeded:
                print(f"   ID={item.key}: {item.error_message}")
                
except Exception as e:
    print(f"❌ 업로드 실패: {str(e)}")


✅ 인덱스 업데이트 완료!
   성공: 10개
   실패: 0개


---
## 8️⃣ 업데이트 검증

업데이트된 문서를 조회하여 증강된 벡터가 정상적으로 저장되었는지 확인합니다.

In [15]:
# 샘플 문서 조회
sample_id = str(df_sample.iloc[0]['id'])

try:
    doc = search_client.get_document(key=sample_id)
    
    print(f"✅ 문서 조회 성공!")
    print(f"\n📋 문서 ID: {doc['id']}")
    print(f"   제품명: {doc.get('title', 'N/A')[:50]}...")
    print(f"   브랜드: {doc.get('brand', 'N/A')}")
    print(f"   카테고리: {doc.get('category', 'N/A')}")
    
    # 벡터 필드 확인
    has_content_vector = 'content_vector' in doc and doc['content_vector'] is not None
    
    print(f"\n🎯 content_vector:")
    print(f"   상태: {'✅ 업데이트됨' if has_content_vector else '❌ 없음'}")
    if has_content_vector:
        print(f"   차원: {len(doc['content_vector'])}")
        print(f"   샘플: [{doc['content_vector'][0]:.6f}, {doc['content_vector'][1]:.6f}, ...]")
    
except Exception as e:
    print(f"❌ 문서 조회 실패: {str(e)}")

✅ 문서 조회 성공!

📋 문서 ID: 1
   제품명: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   브랜드: 압소바
   카테고리: 유아동

🎯 content_vector:
   상태: ✅ 업데이트됨
   차원: 1536
   샘플: [0.047271, 0.018647, ...]


---
## 8️⃣-2 증강 효과 검증 (벡터 비교)

**기존 벡터 vs 증강된 벡터**의 차이를 정량적으로 비교하여 증강이 실제로 효과가 있는지 확인합니다.

In [16]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ✅ 6항에서 저장한 기존 벡터(df_sample['existing_vector'])와 증강된 벡터(df_sample['enriched_vector']) 비교
# 7항에서 인덱스가 업데이트되었으므로, 백업본을 사용하여 비교합니다.
print("📊 기존 벡터와 증강 벡터 비교 분석")
print("="*80)
print("📌 기존 벡터: 7항 전에 인덱스에서 백업한 벡터")
print("📌 증강 벡터: 증강된 콘텐츠로 새로 생성한 벡터")
print()

sample_docs = []
for idx, row in df_sample.head(5).iterrows():
    if row['existing_vector'] is not None and row['enriched_vector'] is not None:
        sample_docs.append({
            'id': str(row['id']),
            'title': row['title'],
            'existing_vector': row['existing_vector'],     # ✅ 저장된 기존 벡터
            'enriched_vector': row['enriched_vector'],     # ✅ 증강된 벡터
        })

print(f"✅ {len(sample_docs)}개 문서 준비 완료 (기존 벡터 백업본 사용)\n")

# 코사인 유사도 계산
similarity_results = []
for doc in sample_docs:
    if doc['existing_vector'] is not None and doc['enriched_vector'] is not None:
        existing_vec = np.array(doc['existing_vector']).reshape(1, -1)
        enriched_vec = np.array(doc['enriched_vector']).reshape(1, -1)
        
        similarity = cosine_similarity(existing_vec, enriched_vec)[0][0]
        
        similarity_results.append({
            'id': doc['id'],
            'title': doc['title'][:40],
            'similarity': similarity
        })

# 결과 출력
print("🔍 코사인 유사도 분석 (기존 벡터 vs 증강 벡터)")
print("-"*80)
print(f"{'제품명':<40} {'유사도':<10} {'해석':<20}")
print("-"*80)

similarities = []
for result in similarity_results:
    sim = result['similarity']
    similarities.append(sim)
    
    # 유사도 해석
    if sim > 0.95:
        interpretation = "증강 미미 ⚠️"
    elif sim > 0.90:
        interpretation = "약간의 증강"
    elif sim > 0.85:
        interpretation = "중간 정도 증강"
    elif sim > 0.80:
        interpretation = "뚜렷한 증강 ✅"
    else:
        interpretation = "큰 증강 ⭐"
    
    print(f"{result['title']:<40} {sim:<10.4f} {interpretation:<20}")

# 통계
print("-"*80)
if similarities:
    print(f"평균 유사도: {np.mean(similarities):.4f}")
    print(f"최소 유사도: {np.min(similarities):.4f}")
    print(f"최대 유사도: {np.max(similarities):.4f}")
    print(f"표준편차: {np.std(similarities):.4f}")
    print()
    
    avg_sim = np.mean(similarities)
    if avg_sim > 0.95:
        print("📌 결론: 기존 벡터와 유사합니다. 증강 효과가 미미합니다.")
    elif avg_sim > 0.90:
        print("📌 결론: 일부 정보가 추가되었습니다. 증강 효과가 있습니다.")
    elif avg_sim > 0.85:
        print("📌 결론: 상당한 정보가 추가되었습니다. 증강 효과가 명확합니다. ✅")
    else:
        print("📌 결론: 기존 벡터와 크게 달라졌습니다. 새로운 정보가 많이 추가되었습니다. ⭐")


📊 기존 벡터와 증강 벡터 비교 분석
📌 기존 벡터: 7항 전에 인덱스에서 백업한 벡터
📌 증강 벡터: 증강된 콘텐츠로 새로 생성한 벡터

✅ 5개 문서 준비 완료 (기존 벡터 백업본 사용)

🔍 코사인 유사도 분석 (기존 벡터 vs 증강 벡터)
--------------------------------------------------------------------------------
제품명                                      유사도        해석                  
--------------------------------------------------------------------------------
압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)    0.8549     중간 정도 증강            
[압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)     0.8479     뚜렷한 증강 ✅            
블루독베이비[hu] WH)블루독남아손수건SET 41170-006-01 손 0.8509     중간 정도 증강            
블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 0.8982     중간 정도 증강            
(에뜨와) (07P083101) (임신출산백일선물)코니딸랑이세트(6종)  0.8664     중간 정도 증강            
--------------------------------------------------------------------------------
평균 유사도: 0.8637
최소 유사도: 0.8479
최대 유사도: 0.8982
표준편차: 0.0184

📌 결론: 상당한 정보가 추가되었습니다. 증강 효과가 명확합니다. ✅


### 🔬 대조군 검증 (코사인 유사도 함수 검증)

대조군(df_holdout_tail)을 사용하여 **코사인 유사도 비교 함수가 제대로 작동하는지** 검증합니다.  
4항에서 백업한 기존 벡터와 현재 인덱스의 벡터를 비교하여 동일한지 확인합니다 (유사도 ~1.0).

In [22]:
# ✅ 4항에서 저장한 기존 벡터(df_holdout_tail['existing_vector'])와 현재 인덱스 벡터 비교
# 대조군은 증강하지 않았으므로, 벡터가 동일해야 함
print("📊 대조군 벡터 검증 (df_holdout_tail - 증강 안 함)")
print("="*80)
print("📌 4항 백업 벡터: 증강 전 인덱스에서 백업한 벡터")
print("📌 현재 벡터: 실험군 증강 후 인덱스 상태 (대조군은 변경 안 됨)")
print()

# 현재 인덱스에서 벡터 조회
print("💾 대조군 현재 벡터 조회...")
holdout_current_vectors = []
for idx, row in tqdm(df_holdout_tail.iterrows(), total=len(df_holdout_tail), desc="현재 벡터 조회"):
    try:
        doc = search_client.get_document(key=str(row['id']))
        if 'content_vector' in doc and doc['content_vector'] is not None:
            holdout_current_vectors.append(doc['content_vector'])
        else:
            holdout_current_vectors.append(None)
    except Exception as e:
        print(f"⚠️  문서 조회 실패 (ID={row['id']}): {str(e)[:40]}")
        holdout_current_vectors.append(None)

df_holdout_tail['current_vector'] = holdout_current_vectors
print(f"✅ 현재 벡터 조회 완료: {sum(1 for v in holdout_current_vectors if v is not None)}/{len(df_holdout_tail)}개\n")

# 코사인 유사도 계산 (기존 벡터 vs 현재 벡터)
print("🔍 코사인 유사도 분석 (4항 백업 벡터 vs 현재 벡터)")
print("-"*80)
print(f"{'제품명':<40} {'유사도':<10} {'해석':<20}")
print("-"*80)

holdout_comparison = []
for idx, row in df_holdout_tail.iterrows():
    if row['existing_vector'] is not None and row['current_vector'] is not None:
        vec_existing = np.array(row['existing_vector']).reshape(1, -1)
        vec_current = np.array(row['current_vector']).reshape(1, -1)
        
        similarity = cosine_similarity(vec_existing, vec_current)[0][0]
        
        holdout_comparison.append({
            'id': str(row['id']),
            'title': row['title'][:40],
            'similarity': similarity,
            'is_identical': similarity > 0.9999
        })
        
        # 유사도 해석
        interpretation = "✅ 동일" if similarity > 0.9999 else "⚠️ 변경됨"
        print(f"{row['title'][:40]:<40} {similarity:<10.8f} {interpretation:<20}")

# 통계
print("-"*80)
if holdout_comparison:
    similarities = [r['similarity'] for r in holdout_comparison]
    identical_count = sum(1 for r in holdout_comparison if r['is_identical'])
    
    print(f"평균 유사도: {np.mean(similarities):.8f}")
    print(f"최소 유사도: {np.min(similarities):.8f}")
    print(f"최대 유사도: {np.max(similarities):.8f}")
    print(f"동일 판정: {identical_count}/{len(holdout_comparison)}개")
    print()
    
    if np.mean(similarities) > 0.9999:
        print("✅ 검증 성공: 대조군 벡터가 동일합니다.")
        print("✅ 코사인 유사도 함수가 정확하게 작동합니다.")
    else:
        print("⚠️ 검증 실패: 대조군 벡터에 예상치 못한 변화가 있습니다.")
        print("   (증강하지 않은 데이터인데 벡터가 변경됨)")
else:
    print("⚠️ 비교 가능한 데이터가 없습니다.")


📊 대조군 벡터 검증 (df_holdout_tail - 증강 안 함)
📌 4항 백업 벡터: 증강 전 인덱스에서 백업한 벡터
📌 현재 벡터: 실험군 증강 후 인덱스 상태 (대조군은 변경 안 됨)

💾 대조군 현재 벡터 조회...


현재 벡터 조회: 100%|██████████| 10/10 [00:01<00:00,  7.69it/s]

✅ 현재 벡터 조회 완료: 10/10개

🔍 코사인 유사도 분석 (4항 백업 벡터 vs 현재 벡터)
--------------------------------------------------------------------------------
제품명                                      유사도        해석                  
--------------------------------------------------------------------------------
[아디다스코리아공식] JS4939 아디제로 보스턴 13           1.00000000 ✅ 동일                
[미꼬][NEW/온라인 전용]플로우 펜던트 브레이슬릿 HDMI0623-1 1.00000000 ✅ 동일                
폴로 랄프 로렌 남성 풀 그레인 레더 벨트(MAPOBLT0F3203002 1.00000000 ✅ 동일                
헤지스핸드백 HIUM5F110BK Toile de Jouy 블랙 경량 양 1.00000000 ✅ 동일                
[공식] 미도 오션스타 월드타이머 M0268301603000        1.00000000 ✅ 동일                
마이크로 러기지 이지 + 쿠션                         1.00000000 ✅ 동일                
[골든듀] 펄시피아미뇽2(D) 펜던트(체인제외)_J1250-0018    1.00000000 ✅ 동일                
[공식/10년보증]프리미엄 마와 옷걸이 굿페어 세트 (25개구성) - 화 1.00000000 ✅ 동일                
[공식/10년보증][추가증정] 프리미엄 마와 옷장정리 싱글세트 (26개구 1.00000000 ✅ 동일                
도러블 펫아이스크림 5개세트                          1.00000000 

In [27]:
# 콘텐츠 변화 시각화
print("\n" + "="*80)
print("📄 콘텐츠 증강 내용 비교")
print("="*80)

sample_idx = 0
if len(df_sample) > sample_idx:
    sample_row = df_sample.iloc[sample_idx]
    
    print(f"\n📌 제품: {sample_row['title'][:50]}")
    print("-"*80)
    
    # 기존 콘텐츠 (메타데이터만)
    print("[기존 콘텐츠] (벡터화 대상)")
    print("-"*80)
    original = f"""**제품명**: {sample_row['title']}
**브랜드**: {sample_row['brand']}
**카테고리**: {sample_row['category']}"""
    
    print(original)
    original_length = len(original)
    
    print(f"\n길이: {original_length}자")
    
    # 증강된 콘텐츠
    print("\n" + "-"*80)
    print("[증강된 콘텐츠] (벡터화 대상)")
    print("-"*80)
    enriched = sample_row['enriched_content']
    print(enriched)
    
    enriched_length = len(enriched)
    print(f"\n길이: {enriched_length}자")
    
    # 비교
    print("\n" + "-"*80)
    print("[증강 분석]")
    print("-"*80)
    increase = enriched_length - original_length
    increase_percent = (increase / original_length) * 100 if original_length > 0 else 0
    
    print(f"📊 길이 증가: {original_length}자 → {enriched_length}자")
    print(f"📊 증가량: {increase}자 (+{increase_percent:.1f}%)")
    
    # 추가된 내용
    if sample_row['image_analysis']:
        print(f"\n✨ 추가된 정보 (이미지 분석):")
        print(f"   {sample_row['image_analysis'][:100]}...")



📄 콘텐츠 증강 내용 비교

📌 제품: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
--------------------------------------------------------------------------------
[기존 콘텐츠] (벡터화 대상)
--------------------------------------------------------------------------------
**제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동

길이: 73자

--------------------------------------------------------------------------------
[증강된 콘텐츠] (벡터화 대상)
--------------------------------------------------------------------------------
**제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동
**제품특징**: 이 제품은 부드럽고 파스텔톤의 연한 녹색, 파랑, 주황색 등 아기 눈에 편안한 색상 조합으로 디자인되어 신생아에게 적합합니다. 소재는 아기 손에 안전한 무독성 플라스틱과 부드러운 천 재질이 혼합되어 있어 촉감과 안전성을 모두 고려한 점이 돋보입니다. 신생아 백일 선물용으로 적합하며, 시각적 자극과 촉각 발달을 돕는 딸랑이와 치발기 세트로 활용도가 높습니다. 타겟 고객은 신생아 부모 및 친지로, 안전하고 감각 발달에 좋은 유아용 장난감을 찾는 이들에게 어필할 수 있습니다. ‘신생아 딸랑이’, ‘백일 선물’, ‘안전한 유아 장난감’ 키워드로 마케팅하면 효과적입니다.

길이: 401자

--------------------------------------------------------------------------------


---
## 9️⃣ 전체 데이터 처리 (선택)

⚠️ **프로덕션 적용**: 샘플 검증이 완료되었으므로, 이제 전체 데이터에 적용할 수 있습니다.

전체 247개 제품에 동일한 프로세스를 적용합니다:
1. 이미지 분석 (GPT Vision)
2. 콘텐츠 증강
3. 벡터 생성
4. 인덱스 업데이트

⚠️ **예상 소요 시간**: 약 15-20분  
⚠️ **예상 비용**: GPT Vision API + Embedding API 호출 비용

In [31]:
# 전체 데이터 처리 여부 선택
PROCESS_ALL = True  # True로 변경하면 전체 처리

if PROCESS_ALL:
    print("🚀 전체 데이터 처리 시작!")
    print(f"   대상: {len(df)}개 제품")
    print(f"   예상 소요 시간: 약 {len(df) * 3}초 ~ {len(df) * 5}초")
    print("\n⚠️  비용 및 시간 소요에 주의하세요!\n")
    
    # 전체 데이터 복사
    df_full = df.copy()
    
    # 1단계: 이미지 분석
    print("1️⃣ 전체 이미지 분석 시작...")
    full_image_analyses = []
    full_success_count = 0
    full_fail_count = 0
    
    for idx, row in tqdm(df_full.iterrows(), total=len(df_full), desc="이미지 분석"):
        analysis = analyze_product_image(
            image_url=row['image_link'],
            title=row['title'],
            category=row['category']
        )
        
        if analysis:
            full_success_count += 1
        else:
            full_fail_count += 1
            analysis = ""
        
        full_image_analyses.append(analysis)
        time.sleep(1)  # Rate limit 방지
    
    df_full['image_analysis'] = full_image_analyses
    print(f"✅ 이미지 분석 완료: 성공 {full_success_count}개, 실패 {full_fail_count}개\n")
    
    # 2단계: 증강 콘텐츠 생성
    print("2️⃣ 증강 콘텐츠 생성 중...")
    full_enriched_contents = []
    for idx, row in df_full.iterrows():
        enriched = create_enriched_content(row, row['image_analysis'])
        full_enriched_contents.append(enriched)
    
    df_full['enriched_content'] = full_enriched_contents
    print(f"✅ 증강 콘텐츠 생성 완료: {len(full_enriched_contents)}개\n")
    
    # 3단계: 벡터 생성
    print("3️⃣ 증강된 콘텐츠 임베딩 생성 중...")
    full_enriched_vectors = []
    for i in tqdm(range(0, len(df_full), BATCH_SIZE), desc="임베딩 생성"):
        batch = df_full['enriched_content'].iloc[i:i+BATCH_SIZE].tolist()
        vectors = get_embeddings_batch(batch)
        full_enriched_vectors.extend(vectors)
        time.sleep(0.5)
    
    df_full['enriched_vector'] = full_enriched_vectors
    full_vector_success = sum(1 for v in full_enriched_vectors if v is not None)
    print(f"✅ 임베딩 생성 완료: {full_vector_success}/{len(df_full)}개\n")
    
    # 4단계: 인덱스 업데이트
    print("4️⃣ 인덱스 업데이트 중...")
    full_documents = []
    for idx, row in df_full.iterrows():
        if row['enriched_vector'] is not None:
            doc = {
                "id": str(row['id']),
                "content_vector": row['enriched_vector']
            }
            full_documents.append(doc)
    
    print(f"📤 업로드 준비: {len(full_documents)}개 문서")
    
    # 배치로 업로드 (한 번에 1000개씩)
    UPLOAD_BATCH_SIZE = 1000
    total_success = 0
    total_failed = 0
    
    for i in tqdm(range(0, len(full_documents), UPLOAD_BATCH_SIZE), desc="인덱스 업데이트"):
        batch_docs = full_documents[i:i+UPLOAD_BATCH_SIZE]
        try:
            result = search_client.merge_or_upload_documents(documents=batch_docs)
            batch_success = sum(1 for item in result if item.succeeded)
            batch_failed = sum(1 for item in result if not item.succeeded)
            total_success += batch_success
            total_failed += batch_failed
        except Exception as e:
            print(f"⚠️  배치 {i//UPLOAD_BATCH_SIZE + 1} 업로드 실패: {str(e)[:50]}")
            total_failed += len(batch_docs)
        
        time.sleep(0.5)
    
    print(f"\n{'='*60}")
    print(f"✅ 전체 데이터 처리 완료!")
    print(f"{'='*60}")
    print(f"   이미지 분석: {full_success_count}/{len(df_full)}개")
    print(f"   벡터 생성: {full_vector_success}/{len(df_full)}개")
    print(f"   인덱스 업데이트: {total_success}개 (실패: {total_failed}개)")
    print(f"{'='*60}")
    
else:
    print("ℹ️  전체 데이터 처리를 건너뜁니다.")
    print("   전체 처리를 원하면 PROCESS_ALL = True로 설정하세요.")

🚀 전체 데이터 처리 시작!
   대상: 247개 제품
   예상 소요 시간: 약 741초 ~ 1235초

⚠️  비용 및 시간 소요에 주의하세요!

1️⃣ 전체 이미지 분석 시작...


이미지 분석:  25%|██▌       | 62/247 [04:53<14:12,  4.61s/it]

⚠️  이미지 분석 에러, 2초 후 재시도... (1/3)
⚠️  이미지 분석 에러, 4초 후 재시도... (2/3)
❌ 이미지 분석 실패: Error code: 400 - {'error': {'code': 'BadRequest',...


이미지 분석:  44%|████▍     | 109/247 [08:39<10:57,  4.77s/it]


KeyboardInterrupt: 

---
## 9️⃣-2 비즈니스 시나리오 검색 테스트

실제 비즈니스 상황에서 발생할 수 있는 다양한 검색 쿼리로 증강 효과를 검증합니다.

In [ ]:
from azure.search.documents.models import VectorizedQuery

def business_search(query_text, top=5, show_details=True):
    """
    비즈니스 시나리오 검색
    
    Args:
        query_text: 검색 쿼리
        top: 반환할 결과 수
        show_details: 상세 정보 출력 여부
    """
    # 쿼리 임베딩
    query_vector = get_embeddings_batch([query_text])[0]
    
    if query_vector is None:
        print("❌ 쿼리 임베딩 실패")
        return None
    
    # 벡터 검색
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=top,
        fields="content_vector"
    )
    
    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        top=top
    )
    
    if show_details:
        print(f"\n🔍 검색어: '{query_text}'")
        print("-" * 80)
        print(f"{'순위':<5} {'제품명':<45} {'브랜드':<15} {'점수':<8}")
        print("-" * 80)
        
        result_list = []
        for idx, result in enumerate(results, 1):
            title = result['title'][:40]
            brand = result.get('brand', 'N/A')[:12]
            score = result['@search.score']
            
            print(f"{idx:<5} {title:<45} {brand:<15} {score:<8.4f}")
            result_list.append(result)
        
        return result_list
    else:
        return list(results)

print("✅ 비즈니스 검색 함수 정의 완료")

In [ ]:
print("="*80)
print("🛍️ 비즈니스 시나리오 검색 테스트")
print("="*80)
print()

# 시나리오 1: 색상 및 스타일 기반 검색
print("📌 시나리오 1: 색상 및 스타일 기반 검색")
print("   고객이 원하는 색상과 스타일로 제품을 찾습니다.")
business_search("파란색 귀여운 디자인의 아기 용품", top=3)
time.sleep(0.5)

print("\n" + "="*80 + "\n")

# 시나리오 2: 용도 및 기능 기반 검색
print("📌 시나리오 2: 용도 및 기능 기반 검색")
print("   특정 용도나 기능을 가진 제품을 찾습니다.")
business_search("운동할 때 편안하고 통기성 좋은 옷", top=3)
time.sleep(0.5)

print("\n" + "="*80 + "\n")

# 시나리오 3: 분위기 및 느낌 기반 검색
print("📌 시나리오 3: 분위기 및 느낌 기반 검색")
print("   추상적인 느낌이나 분위기로 제품을 찾습니다.")
business_search("고급스럽고 세련된 느낌의 가방", top=3)
time.sleep(0.5)

print("\n" + "="*80 + "\n")

# 시나리오 4: 타겟 고객층 기반 검색
print("📌 시나리오 4: 타겟 고객층 기반 검색")
print("   특정 고객층을 위한 제품을 찾습니다.")
business_search("신생아 선물용 부드러운 제품", top=3)
time.sleep(0.5)

print("\n" + "="*80 + "\n")

# 시나리오 5: 상황 및 시나리오 기반 검색
print("📌 시나리오 5: 상황 및 시나리오 기반 검색")
print("   특정 상황이나 시나리오에 맞는 제품을 찾습니다.")
business_search("출근용 깔끔하고 단정한 옷", top=3)
time.sleep(0.5)

print("\n" + "="*80)
print("✅ 비즈니스 시나리오 검색 테스트 완료!")
print("="*80)
print("""
💡 증강 효과 분석:
- 이미지 분석으로 추출한 색상, 디자인, 용도 정보가 검색에 활용됨
- 기존에는 불가능했던 추상적/시각적 검색어로도 관련 제품 검색 가능
- 고객의 자연스러운 표현으로 제품을 찾을 수 있어 검색 편의성 향상
""")

---
## 🔟 정리 및 다음 단계

### ✅ 완료한 작업

1. **이미지 분석**: GPT Vision으로 제품 이미지 분석
2. **콘텐츠 증강**: 기존 텍스트에 이미지 분석 결과 추가
3. **벡터 재생성**: 증강된 콘텐츠로 임베딩 생성
4. **인덱스 업데이트**: content_vector 필드 업데이트

### 📊 증강 전 vs 증강 후

| 항목 | 증강 전 | 증강 후 |
|------|--------|--------|
| **content_text 구성** | 제품명 + 브랜드 + 카테고리 | + 제품특징 (이미지 분석) |
| **검색 가능 정보** | 기본 메타데이터 | + 색상, 디자인, 용도, 타겟 고객 |
| **검색 품질** | 기본 | 향상 |

### 💡 활용 시나리오

- **시각적 특징 검색**: "파란색 가벼운 백팩" → 이미지에서 추출한 색상/특징 매칭
- **용도 기반 검색**: "출근용 세련된 옷" → 이미지 분석에서 추출한 사용 시나리오 매칭
- **마케팅 키워드 검색**: 고객이 사용할 법한 표현으로 검색 가능

### ⚠️ 주의사항

- **비용**: GPT Vision API 호출 비용 발생
- **시간**: 이미지 분석은 텍스트 처리보다 시간 소요
- **품질 관리**: AI 분석 결과의 정확성 검증 필요

### 🚀 전체 데이터 처리

전체 247개 제품을 처리하려면:
1. `SAMPLE_SIZE = len(df)` 로 변경
2. 예상 소요 시간: 약 15-20분
3. 예상 비용: GPT Vision + Embedding API 비용

In [ ]:
print("✅ 데이터 증강 실습 완료!")
print("\n📋 다음 단계:")
print("   1. 전체 데이터 처리 (선택)")
print("   2. 08-ai-skills: Custom Skills로 비즈니스 로직 적용")
print("   3. 09-foundryiq: FoundryIQ 통합")